# 04 - Official Early-Exit Vision Transformer (BranchyNet / LGViT Standard) Baseline
Official PyTorch implementation and evaluation of the **Early-Exit Vision Transformer (BranchyNet / LGViT standard)** on the Imagenette benchmark with standardized ImageNet-C mathematical corruptions.

### Scientific Purpose of this Baseline:
1. **Homogeneous vs. Heterogeneous Adaptive Inference:** Compare whether attaching early exit heads inside a single Vision Transformer backbone can match or outperform our **First-Touch Heterogeneous Gumbel Router** (which routes dynamically between SqueezeNet 1.0 and DeiT-S-16).
2. **Exact Preprocessing from `routing_beats_latency_without_gumbel_v8.ipynb`:** Uses the identical `expert_eval_transform` (Resize to 224 -> corrupt -> Resize to 384 -> ToTensor -> Normalize ImageNet mean/std).
3. **Official Architecture Standards (BranchyNet / LGViT / CF-ViT):**
   * Backbone: `deit_small_patch16_224` (12 Transformer Encoder Blocks, $D=384, L=12$).
   * **Exit 1:** Attached after **Block 4** (Depth 4/12 — Early Stage).
   * **Exit 2:** Attached after **Block 8** (Depth 8/12 — Mid Stage).
   * **Exit 3:** Final Output after **Block 12** (Depth 12/12 — Full Depth).
   * Exit Heads: Standard `LayerNorm(384) -> Linear(384, 10)` attached to the `[CLS]` visual token.


## 1. Setup, Environment, and Mathematical Corruptions


In [1]:
import os, sys, gc, glob, random, math, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.models as models
import torchvision.datasets as torchvision_datasets
from tqdm.auto import tqdm
import timm

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")

# ---- Paths & Configuration --------------------------------------------------
if 'PROJECT_ROOT' not in globals() or not os.path.exists(os.path.join(globals().get('PROJECT_ROOT', ''), 'outputs_imagenette_math')):
    if os.path.exists(r"d:\Saqib\RoutingBeatsLatency\routing_beats_latency_revision2\outputs_imagenette_math"):
        PROJECT_ROOT = r"d:\Saqib\RoutingBeatsLatency\routing_beats_latency_revision2"
    elif os.path.exists("outputs_imagenette_math"):
        PROJECT_ROOT = os.path.abspath(".")
    else:
        PROJECT_ROOT = os.path.abspath("..")

DATA_DIR       = os.path.join(PROJECT_ROOT, "data", "imagenette")
TRAIN_DIR      = os.path.join(DATA_DIR, "train")
VAL_DIR        = os.path.join(DATA_DIR, "val")

OUTPUT_DIR     = os.path.normpath(os.path.join(PROJECT_ROOT, "outputs_imagenette_math"))
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
CACHE_DIR      = os.path.join(OUTPUT_DIR, "cache")
PLOTS_DIR      = os.path.join(OUTPUT_DIR, "plots")
RESULTS_XLSX   = os.path.join(OUTPUT_DIR, "results_imagenette_math.xlsx")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES    = 10
IMG_SIZE       = 384
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
SUPERCLASS_MAP = torch.arange(NUM_CLASSES, dtype=torch.long)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# ---- Standard ImageNet-C Mathematical Corruptions ---------------------------
def apply_gaussian_blur(img, severity):
    sigmas = {"clean": 0, "mild": 1.0, "moderate": 2.0, "severe": 3.5}
    sigma = sigmas.get(severity, 0)
    if sigma == 0: return img
    ksize = int(2 * math.ceil(2 * sigma) + 1)
    return cv2.GaussianBlur(img, (ksize, ksize), sigma)

def apply_gaussian_noise(img, severity):
    sigmas = {"clean": 0, "mild": 15, "moderate": 30, "severe": 60}
    sigma = sigmas.get(severity, 0)
    if sigma == 0: return img
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def apply_jpeg_compression(img, severity):
    qualities = {"clean": 100, "mild": 50, "moderate": 25, "severe": 10}
    q = qualities.get(severity, 100)
    if q == 100: return img
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), q]
    _, encimg = cv2.imencode('.jpg', img, encode_param)
    return cv2.imdecode(encimg, 1)

def apply_contrast_reduction(img, severity):
    factors = {"clean": 1.0, "mild": 0.6, "moderate": 0.35, "severe": 0.15}
    factor = factors.get(severity, 1.0)
    if factor == 1.0: return img
    mean = np.mean(img, axis=(0, 1), keepdims=True)
    return np.clip((img - mean) * factor + mean, 0, 255).astype(np.uint8)

def apply_speckle_noise(img, severity):
    sigmas = {"clean": 0, "mild": 0.1, "moderate": 0.25, "severe": 0.5}
    sigma = sigmas.get(severity, 0)
    if sigma == 0: return img
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) * (1 + noise), 0, 255).astype(np.uint8)

def apply_pixelate(img, severity):
    factors = {"clean": 1.0, "moderate": 0.25, "severe": 0.10}
    factor = factors.get(severity, 1.0)
    if factor == 1.0: return img
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * factor)), max(1, int(h * factor))), interpolation=cv2.INTER_NEAREST)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)

def apply_impulse_noise(img, severity):
    probs = {"clean": 0.0, "moderate": 0.08, "severe": 0.20}
    p = probs.get(severity, 0.0)
    if p == 0.0: return img
    res = img.copy()
    num_salt = np.ceil(p * img.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in img.shape[:2]]
    res[tuple(coords)] = 255
    num_pepper = np.ceil(p * img.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in img.shape[:2]]
    res[tuple(coords)] = 0
    return res

SEEN_CORRUPTIONS = {
    "blur": apply_gaussian_blur,
    "noise": apply_gaussian_noise,
    "jpeg": apply_jpeg_compression,
    "contrast": apply_contrast_reduction,
    "speckle": apply_speckle_noise,
}

UNSEEN_CORRUPTIONS = {
    "pixelate": apply_pixelate,
    "impulse_noise": apply_impulse_noise,
}

# Preprocessing transform identical to routing_beats_latency_without_gumbel_v8.ipynb
expert_eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

print("Setup & corruptions configured successfully.")

PyTorch Version : 2.6.0+cu124
CUDA Available  : True
Device Name     : NVIDIA RTX 4000 Ada Generation
Setup & corruptions configured successfully.


## 2. Deterministic Train/Test Dataset Partitioning
Follows the exact 80/20 train/test split on `data/imagenette/train` with `seed=42`.
* **Train Split:** 7,575 images (80%).
* **Test Split:** 1,894 images (20%).


In [2]:
def get_test_samples(train_dir):
    clean_ds = torchvision_datasets.ImageFolder(train_dir, transform=None)
    generator = torch.Generator().manual_seed(42)
    train_len = int(0.8 * len(clean_ds))
    test_len = len(clean_ds) - train_len
    _, test_ds = torch.utils.data.random_split(clean_ds, [train_len, test_len], generator=generator)
    return [clean_ds.samples[i] for i in test_ds.indices]

def get_train_samples(train_dir):
    clean_ds = torchvision_datasets.ImageFolder(train_dir, transform=None)
    generator = torch.Generator().manual_seed(42)
    train_len = int(0.8 * len(clean_ds))
    test_len = len(clean_ds) - train_len
    train_ds, _ = torch.utils.data.random_split(clean_ds, [train_len, test_len], generator=generator)
    return [clean_ds.samples[i] for i in train_ds.indices]

print(f"Train samples : {len(get_train_samples(TRAIN_DIR)):,}")
print(f"Test samples  : {len(get_test_samples(TRAIN_DIR)):,}")


Train samples : 7,575
Test samples  : 1,894


## 3. Official Early-Exit Vision Transformer Architecture (`EarlyExitDeiTS16`)
We attach 2 intermediate early-exit classification heads to the 12-block `deit_small_patch16_224` backbone matching standard BranchyNet / LGViT specifications:
* **Exit 1:** Attached after **Block 4** (Depth 4/12 — Early Stage)
* **Exit 2:** Attached after **Block 8** (Depth 8/12 — Mid Stage)
* **Exit 3:** Final Output after **Block 12** (Depth 12/12 — Full Depth)


In [3]:
class EarlyExitDeiTS16(nn.Module):
    def __init__(self, num_classes=10, img_size=384, pretrained=True):
        super().__init__()
        self.base_model = timm.create_model(
            "deit_small_patch16_224",
            pretrained=pretrained,
            num_classes=num_classes,
            img_size=img_size
        )
        embed_dim = self.base_model.embed_dim # 384
        
        # Exit 1 Head (after block 4)
        self.exit1_norm = nn.LayerNorm(embed_dim)
        self.exit1_head = nn.Linear(embed_dim, num_classes)
        
        # Exit 2 Head (after block 8)
        self.exit2_norm = nn.LayerNorm(embed_dim)
        self.exit2_head = nn.Linear(embed_dim, num_classes)
        
        self.num_classes = num_classes
        self.embed_dim = embed_dim

    def forward_features_partial(self, x, start_block=0, end_block=12):
        if start_block == 0:
            x = self.base_model.patch_embed(x)
            x = self.base_model._pos_embed(x)
            x = self.base_model.norm_pre(x)
        
        for i in range(start_block, end_block):
            x = self.base_model.blocks[i](x)
            
        return x

    def forward_train(self, x):
        # Stage 1: Blocks 0..3
        x4 = self.forward_features_partial(x, 0, 4)
        cls_4 = x4[:, 0]
        logits1 = self.exit1_head(self.exit1_norm(cls_4))
        
        # Stage 2: Blocks 4..7
        x8 = self.forward_features_partial(x4, 4, 8)
        cls_8 = x8[:, 0]
        logits2 = self.exit2_head(self.exit2_norm(cls_8))
        
        # Stage 3: Blocks 8..11
        x12 = self.forward_features_partial(x8, 8, 12)
        x12 = self.base_model.norm(x12)
        logits3 = self.base_model.head(x12[:, 0])
        
        return logits1, logits2, logits3

    def forward_dynamic(self, x, threshold=0.8):
        # Stage 1: Blocks 0..3
        x4 = self.forward_features_partial(x, 0, 4)
        cls_4 = x4[:, 0]
        logits1 = self.exit1_head(self.exit1_norm(cls_4))
        probs1 = torch.softmax(logits1, dim=-1)
        conf1, _ = probs1.max(dim=-1)
        
        if conf1.item() >= threshold:
            return logits1, 1
            
        # Stage 2: Blocks 4..7
        x8 = self.forward_features_partial(x4, 4, 8)
        cls_8 = x8[:, 0]
        logits2 = self.exit2_head(self.exit2_norm(cls_8))
        probs2 = torch.softmax(logits2, dim=-1)
        conf2, _ = probs2.max(dim=-1)
        
        if conf2.item() >= threshold:
            return logits2, 2
            
        # Stage 3: Blocks 8..11
        x12 = self.forward_features_partial(x8, 8, 12)
        x12 = self.base_model.norm(x12)
        logits3 = self.base_model.head(x12[:, 0])
        return logits3, 3

    def forward(self, x, threshold=None):
        if threshold is None or self.training:
            return self.forward_train(x)
        return self.forward_dynamic(x, threshold)

print("EarlyExitDeiTS16 architecture compiled successfully.")


EarlyExitDeiTS16 architecture compiled successfully.


## 4. Single-Sample Hardware Latency Profiling (Batch Size = 1)
Measures the true operational GPU execution times using hardware-level CUDA Events (20 warmup runs + 200 timed iterations).


In [4]:
@torch.inference_mode()
def benchmark_exit_latencies():
    model = EarlyExitDeiTS16(num_classes=NUM_CLASSES, img_size=IMG_SIZE, pretrained=False).to(DEVICE)
    model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    
    # Warmup
    for _ in range(20):
        _ = model.forward_features_partial(dummy, 0, 4)
        _ = model.forward_features_partial(dummy, 0, 8)
        _ = model.forward_features_partial(dummy, 0, 12)
    torch.cuda.synchronize()
    
    # 1. Exit 1 Latency (Blocks 0..3)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    times_exit1 = []
    for _ in range(200):
        starter.record()
        x4 = model.forward_features_partial(dummy, 0, 4)
        _ = model.exit1_head(model.exit1_norm(x4[:, 0]))
        ender.record()
        torch.cuda.synchronize()
        times_exit1.append(starter.elapsed_time(ender))
        
    # 2. Exit 2 Latency (Blocks 0..7)
    times_exit2 = []
    for _ in range(200):
        starter.record()
        x8 = model.forward_features_partial(dummy, 0, 8)
        _ = model.exit2_head(model.exit2_norm(x8[:, 0]))
        ender.record()
        torch.cuda.synchronize()
        times_exit2.append(starter.elapsed_time(ender))
        
    # 3. Final Exit Latency (Full 12/12)
    times_final = []
    for _ in range(200):
        starter.record()
        x12 = model.forward_features_partial(dummy, 0, 12)
        _ = model.base_model.head(model.base_model.norm(x12)[:, 0])
        ender.record()
        torch.cuda.synchronize()
        times_final.append(starter.elapsed_time(ender))
        
    t_exit1 = float(np.median(times_exit1))
    t_exit2 = float(np.median(times_exit2))
    t_final = float(np.median(times_final))
    
    print("=" * 80)
    print("EARLY-EXIT DEIT-S-16 HARDWARE LATENCY PROFILING (BATCH SIZE = 1)")
    print("=" * 80)
    print(f"  • Exit 1 (Depth 4/12)   : {t_exit1:.2f} ms")
    print(f"  • Exit 2 (Depth 8/12)   : {t_exit2:.2f} ms")
    print(f"  • Exit 3 (Full 12/12)   : {t_final:.2f} ms")
    print("=" * 80)
    return {"Exit_1": t_exit1, "Exit_2": t_exit2, "Exit_3": t_final}

EARLY_EXIT_LATENCIES = benchmark_exit_latencies()


EARLY-EXIT DEIT-S-16 HARDWARE LATENCY PROFILING (BATCH SIZE = 1)
  • Exit 1 (Depth 4/12)   : 1.55 ms
  • Exit 2 (Depth 8/12)   : 3.24 ms
  • Exit 3 (Full 12/12)   : 4.89 ms


## 5. Multi-Exit Joint Training on GPU (50 Epochs, Patience = 5)
Follows the official BranchyNet / LGViT paradigm:
1. Loads the fine-tuned DeiT-S-16 backbone.
2. Extracts visual token embeddings at Block 4 and Block 8 using the exact `expert_eval_transform` (~3 seconds on GPU).
3. Trains the early exit heads across 50 epochs with AdamW, Cosine Annealing, and Early Stopping (Patience = 5) in ~2 seconds.


In [5]:
def finetune_early_exit_vit_official(epochs=50, patience=5, lr=1e-3, batch_size=128):
    save_path = os.path.join(CHECKPOINT_DIR, "EarlyExit_DeiT_S16.pt")
    set_seed(42)
    
    # 1. Initialize Model & Load Fine-Tuned DeiT-S-16 Backbone
    model = EarlyExitDeiTS16(num_classes=NUM_CLASSES, img_size=IMG_SIZE, pretrained=True).to(DEVICE)
    base_ckpt_path = os.path.join(CHECKPOINT_DIR, "DeiT-S-16.pt")
    if os.path.exists(base_ckpt_path):
        print(f"Loading fine-tuned DeiT-S-16 backbone from {base_ckpt_path}...")
        base_ckpt = torch.load(base_ckpt_path, map_location=DEVICE, weights_only=False)
        model.base_model.load_state_dict(base_ckpt["state_dict"], strict=False)
    
    # 2. Fast GPU Batch Extraction of Visual Tokens matching v8 preprocessing
    print("\n[Fast-Extraction] Extracting intermediate visual tokens at Block 4 & Block 8...")
    train_samples = get_train_samples(TRAIN_DIR)
    
    model.eval()
    cls4_list, cls8_list, labels_list = [], [], []
    corrs = list(SEEN_CORRUPTIONS.keys())
    
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=torch.float16):
            for idx in tqdm(range(0, len(train_samples), 64), desc="Extracting Tokens"):
                batch_samples = train_samples[idx:idx+64]
                batch_imgs = []
                batch_labs = []
                for s_idx, (path, label) in enumerate(batch_samples):
                    img = Image.open(path).convert("RGB")
                    img = transforms.Resize((224, 224))(img)
                    arr = np.array(img)
                    rand = random.random()
                    if rand >= 0.60:
                        c = random.choice(corrs)
                        sev = "moderate" if rand < 0.80 else "severe"
                        arr = SEEN_CORRUPTIONS[c](arr, sev)
                    pil_img = Image.fromarray(arr)
                    batch_imgs.append(expert_eval_transform(pil_img))
                    batch_labs.append(SUPERCLASS_MAP[label].item())
                    
                x_tensor = torch.stack(batch_imgs).to(DEVICE)
                x4 = model.forward_features_partial(x_tensor, 0, 4)
                x8 = model.forward_features_partial(x4, 4, 8)
                cls4_list.append(x4[:, 0])
                cls8_list.append(x8[:, 0])
                labels_list.extend(batch_labs)
            
    X4 = torch.cat(cls4_list)
    X8 = torch.cat(cls8_list)
    Y  = torch.tensor(labels_list, dtype=torch.long, device=DEVICE)
    
    # 3. Train Exit Heads with AdamW + Early Stopping (Patience = 5)
    train_dataset = TensorDataset(X4, X8, Y)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    optimizer = optim.AdamW(
        list(model.exit1_norm.parameters()) + list(model.exit1_head.parameters()) +
        list(model.exit2_norm.parameters()) + list(model.exit2_head.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    best_loss = float("inf")
    best_weights = None
    patience_counter = 0
    history = []
    
    print(f"\n" + "=" * 80)
    print(f"STARTING MULTI-EXIT HEAD TRAINING ({epochs} MAX EPOCHS, PATIENCE={patience}, BS={batch_size})")
    print("=" * 80)
    
    for epoch in range(1, epochs + 1):
        model.train()
        tot_loss, tot_c1, tot_c2, total = 0.0, 0, 0, 0
        
        for bx4, bx8, by in train_loader:
            optimizer.zero_grad()
            l1 = model.exit1_head(model.exit1_norm(bx4))
            l2 = model.exit2_head(model.exit2_norm(bx8))
            
            loss1 = criterion(l1, by)
            loss2 = criterion(l2, by)
            loss = 0.5 * loss1 + 0.5 * loss2
            
            loss.backward()
            optimizer.step()
            
            bs = by.size(0)
            total += bs
            tot_loss += loss.item() * bs
            tot_c1 += (l1.argmax(-1) == by).sum().item()
            tot_c2 += (l2.argmax(-1) == by).sum().item()
            
        scheduler.step()
        avg_loss = tot_loss / total
        acc1 = (tot_c1 / total) * 100.0
        acc2 = (tot_c2 / total) * 100.0
        
        history.append({"epoch": epoch, "loss": avg_loss, "acc_exit1": acc1, "acc_exit2": acc2})
        print(f"Epoch {epoch:02d}/{epochs:02d} | Joint Loss: {avg_loss:.4f} | Exit 1 Acc: {acc1:.2f}% | Exit 2 Acc: {acc2:.2f}% | LR: {scheduler.get_last_lr()[0]:.2e}")
        
        # Early Stopping Check (Patience = 5)
        if avg_loss < best_loss - 1e-4:
            best_loss = avg_loss
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n[Early Stopping Triggered] No loss improvement for {patience} consecutive epochs. Stopping at epoch {epoch:02d} (Best Loss: {best_loss:.4f}).")
                break
                
    if best_weights is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_weights.items()})
        
    torch.save({"state_dict": model.state_dict(), "history": history}, save_path)
    pd.DataFrame(history).to_csv(os.path.join(CHECKPOINT_DIR, "EarlyExit_DeiT_S16_history.csv"), index=False)
    print(f"\nSuccessfully saved best EarlyExitDeiTS16 checkpoint to {save_path}")
    model.eval()
    return model

early_exit_model = finetune_early_exit_vit_official(epochs=50, patience=5, lr=1e-3, batch_size=128)


Loading fine-tuned DeiT-S-16 backbone from d:\Saqib\RoutingBeatsLatency\routing_beats_latency_revision2\outputs_imagenette_math\checkpoints\DeiT-S-16.pt...

[Fast-Extraction] Extracting intermediate visual tokens at Block 4 & Block 8...


C:\Users\Saqib\AppData\Local\Temp\ipykernel_23988\3597010720.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Extracting Tokens:   0%|          | 0/119 [00:00<?, ?it/s]


STARTING MULTI-EXIT HEAD TRAINING (50 MAX EPOCHS, PATIENCE=5, BS=128)
Epoch 01/50 | Joint Loss: 1.8393 | Exit 1 Acc: 31.17% | Exit 2 Acc: 68.48% | LR: 9.99e-04
Epoch 02/50 | Joint Loss: 1.2534 | Exit 1 Acc: 49.66% | Exit 2 Acc: 89.25% | LR: 9.96e-04
Epoch 03/50 | Joint Loss: 1.0418 | Exit 1 Acc: 54.28% | Exit 2 Acc: 91.00% | LR: 9.91e-04
Epoch 04/50 | Joint Loss: 0.9214 | Exit 1 Acc: 58.30% | Exit 2 Acc: 92.09% | LR: 9.84e-04
Epoch 05/50 | Joint Loss: 0.8390 | Exit 1 Acc: 60.67% | Exit 2 Acc: 93.19% | LR: 9.76e-04
Epoch 06/50 | Joint Loss: 0.7788 | Exit 1 Acc: 62.79% | Exit 2 Acc: 93.72% | LR: 9.65e-04
Epoch 07/50 | Joint Loss: 0.7321 | Exit 1 Acc: 64.90% | Exit 2 Acc: 94.32% | LR: 9.52e-04
Epoch 08/50 | Joint Loss: 0.6957 | Exit 1 Acc: 65.77% | Exit 2 Acc: 94.57% | LR: 9.38e-04
Epoch 09/50 | Joint Loss: 0.6655 | Exit 1 Acc: 66.89% | Exit 2 Acc: 94.84% | LR: 9.22e-04
Epoch 10/50 | Joint Loss: 0.6402 | Exit 1 Acc: 67.64% | Exit 2 Acc: 95.05% | LR: 9.05e-04
Epoch 11/50 | Joint Loss: 0.6

## 6. Threshold Sweeping & Evaluation on Seen Corruptions (`test_mixed.pt`)
Evaluates the Pareto frontier of Early-Exit DeiT-S-16 across confidence thresholds $t \in [0.30, 1.00]$.


In [6]:
@torch.inference_mode()
def evaluate_early_exit_on_cache(model, cache_path, thresholds=None):
    if thresholds is None:
        thresholds = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.98, 0.99, 1.00]
        
    print(f"\nEvaluating Early-Exit DeiT-S-16 on {os.path.basename(cache_path)}...")
    cache = torch.load(cache_path, map_location="cpu", weights_only=False)
    test_samples = get_test_samples(TRAIN_DIR)
    
    model.eval()
    all_l1, all_l2, all_l3, all_labels = [], [], [], []
    
    with torch.cuda.amp.autocast(dtype=torch.float16):
        for idx in tqdm(range(0, len(test_samples), 64), desc="Evaluating"):
            batch_samples = test_samples[idx:idx+64]
            batch_imgs = []
            batch_labs = []
            for s_idx, (path, label) in enumerate(batch_samples):
                corr = cache["corruptions"][idx + s_idx]
                sev  = cache["severities"][idx + s_idx]
                img = Image.open(path).convert("RGB")
                img = transforms.Resize((224, 224))(img)
                arr = np.array(img)
                if corr != "clean":
                    if corr in SEEN_CORRUPTIONS:
                        arr = SEEN_CORRUPTIONS[corr](arr, sev)
                    elif corr in UNSEEN_CORRUPTIONS:
                        arr = UNSEEN_CORRUPTIONS[corr](arr, sev)
                pil_img = Image.fromarray(arr)
                batch_imgs.append(expert_eval_transform(pil_img))
                batch_labs.append(SUPERCLASS_MAP[label].item())
                
            x_tensor = torch.stack(batch_imgs).to(DEVICE)
            l1, l2, l3 = model(x_tensor)
            all_l1.append(l1.float().cpu())
            all_l2.append(l2.float().cpu())
            all_l3.append(l3.float().cpu())
            all_labels.extend(batch_labs)
        
    logits1 = torch.cat(all_l1)
    logits2 = torch.cat(all_l2)
    logits3 = torch.cat(all_l3)
    labels  = torch.tensor(all_labels)
    
    probs1 = F.softmax(logits1, dim=-1)
    probs2 = F.softmax(logits2, dim=-1)
    conf1, _ = probs1.max(dim=-1)
    conf2, _ = probs2.max(dim=-1)
    
    t1 = EARLY_EXIT_LATENCIES["Exit_1"]
    t2 = EARLY_EXIT_LATENCIES["Exit_2"]
    t3 = EARLY_EXIT_LATENCIES["Exit_3"]
    
    records = []
    for t in thresholds:
        exit1_mask = (conf1 >= t)
        exit2_mask = (~exit1_mask) & (conf2 >= t)
        exit3_mask = (~exit1_mask) & (~exit2_mask)
        
        final_logits = torch.where(
            exit1_mask.unsqueeze(1),
            logits1,
            torch.where(exit2_mask.unsqueeze(1), logits2, logits3)
        )
        
        preds = final_logits.argmax(dim=-1)
        acc = (preds == labels).float().mean().item() * 100.0
        
        pct1 = exit1_mask.float().mean().item() * 100.0
        pct2 = exit2_mask.float().mean().item() * 100.0
        pct3 = exit3_mask.float().mean().item() * 100.0
        
        avg_lat = (pct1/100.0)*t1 + (pct2/100.0)*t2 + (pct3/100.0)*t3
        saved = ((t3 - avg_lat) / t3) * 100.0
        
        records.append({
            "Threshold": t,
            "Accuracy": acc,
            "Latency_ms": avg_lat,
            "Pct_Exit_1": pct1,
            "Pct_Exit_2": pct2,
            "Pct_Exit_3": pct3,
            "Pct_Saved_vs_Full": saved,
        })
        
    df_results = pd.DataFrame(records)
    return df_results

seen_cache_path = os.path.join(CACHE_DIR, "test_mixed.pt")
df_earlyexit_seen = evaluate_early_exit_on_cache(early_exit_model, seen_cache_path)

print("\n" + "=" * 80)
print("EARLY-EXIT DEIT-S-16 EVALUATION ON SEEN CORRUPTIONS (test_mixed.pt)")
print("=" * 80)
print(df_earlyexit_seen.to_string(index=False))



Evaluating Early-Exit DeiT-S-16 on test_mixed.pt...


C:\Users\Saqib\AppData\Local\Temp\ipykernel_23988\167808366.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


EARLY-EXIT DEIT-S-16 EVALUATION ON SEEN CORRUPTIONS (test_mixed.pt)
 Threshold  Accuracy  Latency_ms  Pct_Exit_1  Pct_Exit_2  Pct_Exit_3  Pct_Saved_vs_Full
      0.30 76.768744    1.644821   94.614571    5.332629    0.052798          66.346721
      0.40 82.470959    1.859679   82.629359   16.525871    0.844773          61.950699
      0.50 87.486798    2.134670   68.004227   29.408658    2.587117          56.324342
      0.60 91.288280    2.396869   55.174232   39.440337    5.385428          50.959731
      0.70 93.875396    2.648348   43.136218   48.521647    8.342133          45.814435
      0.80 95.459348    2.891443   32.734951   54.804647   12.460401          40.840659
      0.85 96.145725    3.039023   26.557550   58.342135   15.100317          37.821157
      0.90 96.409714    3.217610   18.848997   63.093978   18.057021          34.167235
      0.95 96.620905    3.448333   11.615628   63.727564   24.656811          29.446627
      0.98 96.673709    3.707849    4.276663   62.8

## 7. Zero-Shot Generalization on Unseen ImageNet-C Corruptions (`test_unseen_mixed.pt`)
Evaluates Early-Exit DeiT-S-16 on completely unseen corruptions (`pixelate` and `impulse_noise`).


In [7]:
unseen_cache_path = os.path.join(CACHE_DIR, "test_unseen_mixed.pt")
df_earlyexit_unseen = evaluate_early_exit_on_cache(early_exit_model, unseen_cache_path)

print("\n" + "=" * 80)
print("EARLY-EXIT DEIT-S-16 EVALUATION ON UNSEEN CORRUPTIONS (test_unseen_mixed.pt)")
print("=" * 80)
print(df_earlyexit_unseen.to_string(index=False))



Evaluating Early-Exit DeiT-S-16 on test_unseen_mixed.pt...


C:\Users\Saqib\AppData\Local\Temp\ipykernel_23988\167808366.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


EARLY-EXIT DEIT-S-16 EVALUATION ON UNSEEN CORRUPTIONS (test_unseen_mixed.pt)
 Threshold  Accuracy  Latency_ms  Pct_Exit_1  Pct_Exit_2  Pct_Exit_3  Pct_Saved_vs_Full
      0.30 33.210137    1.563947   99.366421    0.633580    0.000000          68.001422
      0.40 35.427666    1.626084   95.934528    3.801478    0.263992          66.730082
      0.50 39.968321    1.832577   85.533261   12.302006    2.164731          62.505225
      0.60 44.614571    2.165977   70.855331   21.752904    7.391763          55.683808
      0.70 50.052798    2.558606   56.282997   27.402323   16.314678          47.650568
      0.80 56.494194    3.062335   40.443507   28.880677   30.675820          37.344194
      0.85 59.028512    3.330758   32.523760   28.616685   38.859555          31.852215
      0.90 62.090814    3.638016   23.125660   28.986272   47.888067          25.565682
      0.95 64.308339    4.025697   12.988384   25.976768   61.034846          17.633677
      0.98 65.205914    4.343047    4.6990

## 8. Head-to-Head Comparison: First-Touch Gumbel Router vs. Early-Exit ViT vs. Cascades
Generates comparative Pareto curves and exports results to Excel.


In [8]:
plt.rcParams.update({'font.sans-serif': 'Arial', 'font.family': 'sans-serif', 'axes.edgecolor': '#cccccc', 'axes.linewidth': 0.8})

wb_data = {}
if os.path.exists(RESULTS_XLSX):
    xl = pd.ExcelFile(RESULTS_XLSX)
    for sheet in xl.sheet_names:
        wb_data[sheet] = xl.parse(sheet)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5), dpi=150)

# ---- Plot 1: Seen Corruptions -----------------------------------------------
ax1.grid(True, linestyle="--", alpha=0.4)
sq_lat, deit_lat = 1.19, 4.98
ax1.scatter([sq_lat], [64.36], color="#e74c3c", s=140, zorder=6, label="Static SqueezeNet (64.4%)")
ax1.scatter([deit_lat], [89.76], color="#2ecc71", s=140, zorder=6, label="Static DeiT-S-16 (89.8%)")

# Confidence Cascades
if "Reviewer_Baselines" in wb_data:
    df_b = wb_data["Reviewer_Baselines"]
    casc = df_b[df_b["Category"] == "Cascade"].sort_values("Latency_ms")
    ax1.plot(casc["Latency_ms"], casc["Accuracy (%)"], color="#f39c12", linestyle="-.", marker="^", markersize=7, label="Confidence Cascades (Sequential)")

# Early-Exit DeiT-S-16
ax1.plot(df_earlyexit_seen["Latency_ms"], df_earlyexit_seen["Accuracy"], color="#9b59b6", linestyle="--", marker="s", markersize=7, linewidth=2, label="Early-Exit DeiT-S-16 (BranchyNet-ViT)")

# Proposed First-Touch Gumbel Router
if "Gumbel_Summary" in wb_data:
    df_g = wb_data["Gumbel_Summary"]
    ax1.plot(df_g["Latency_Mean (ms)"], df_g["Accuracy_Mean (%)"], color="#2980b9", linewidth=2.8, marker="o", markersize=8, label="First-Touch Gumbel Router (Ours)")
    for _, r in df_g.iterrows():
        if r['Lambda'] in [0.70, 0.80, 0.90, 0.92]:
            ax1.annotate(f"λ={r['Lambda']:.2f}", (r['Latency_Mean (ms)'], r['Accuracy_Mean (%)']), textcoords="offset points", xytext=(0, 8), ha='center', fontsize=8, fontweight='bold', color="#1f4e79")

ax1.set_xlabel("Mean Inference Latency (ms / query)", fontsize=11, fontweight='bold')
ax1.set_ylabel("Top-1 Accuracy (%) on Seen Corruptions", fontsize=11, fontweight='bold')
ax1.set_title("Seen ImageNet-C Corruptions: First-Touch Router vs. Early-Exit ViT", fontsize=12, fontweight='bold', pad=12)
ax1.legend(loc="lower right", framealpha=0.9)

# ---- Plot 2: Unseen Corruptions ---------------------------------------------
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.scatter([sq_lat], [35.59], color="#e74c3c", s=140, zorder=6, label="Static SqueezeNet (35.6%)")
ax2.scatter([deit_lat], [65.21], color="#2ecc71", s=140, zorder=6, label="Static DeiT-S-16 (65.2%)")

# Early-Exit DeiT on Unseen
ax2.plot(df_earlyexit_unseen["Latency_ms"], df_earlyexit_unseen["Accuracy"], color="#9b59b6", linestyle="--", marker="s", markersize=7, linewidth=2, label="Early-Exit DeiT-S-16 (BranchyNet-ViT)")

# Proposed Gumbel Router on Unseen
if "Unseen_Gumbel_Summary" in wb_data:
    df_ug = wb_data["Unseen_Gumbel_Summary"]
    ax2.plot(df_ug["Mean Latency (ms)"], df_ug["Top-1 Accuracy (%)"], color="#2980b9", linewidth=2.8, marker="o", markersize=8, label="First-Touch Gumbel Router (Ours)")
    for _, r in df_ug.iterrows():
        if r['Lambda'] in [0.70, 0.80, 0.90, 0.92]:
            ax2.annotate(f"λ={r['Lambda']:.2f}", (r['Mean Latency (ms)'], r['Top-1 Accuracy (%)']), textcoords="offset points", xytext=(0, 8), ha='center', fontsize=8, fontweight='bold', color="#1f4e79")

ax2.set_xlabel("Mean Inference Latency (ms / query)", fontsize=11, fontweight='bold')
ax2.set_ylabel("Top-1 Accuracy (%) on Unseen Corruptions", fontsize=11, fontweight='bold')
ax2.set_title("Unseen ImageNet-C Corruptions: Zero-Shot Generalization Comparison", fontsize=12, fontweight='bold', pad=12)
ax2.legend(loc="lower right", framealpha=0.9)

plt.tight_layout()
pareto_plot_path = os.path.join(PLOTS_DIR, "earlyexit_vit_vs_gumbel_pareto.png")
plt.savefig(pareto_plot_path, dpi=300)
plt.close(fig)
print(f"\nSaved comparison plot to {pareto_plot_path}")

# ---- Export to Excel --------------------------------------------------------
if os.path.exists(RESULTS_XLSX):
    with pd.ExcelWriter(RESULTS_XLSX, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_earlyexit_seen.to_excel(writer, sheet_name="EarlyExit_DeiT_Seen", index=False)
        df_earlyexit_unseen.to_excel(writer, sheet_name="EarlyExit_DeiT_Unseen", index=False)
    print(f"Early-Exit DeiT results successfully appended to {RESULTS_XLSX}")



Saved comparison plot to d:\Saqib\RoutingBeatsLatency\routing_beats_latency_revision2\outputs_imagenette_math\plots\earlyexit_vit_vs_gumbel_pareto.png
Early-Exit DeiT results successfully appended to d:\Saqib\RoutingBeatsLatency\routing_beats_latency_revision2\outputs_imagenette_math\results_imagenette_math.xlsx


c:\Users\Saqib\anaconda3\envs\routingbeatslatency\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
